In [1]:
import wrds
db = wrds.Connection(wrds_username='liaojy')

Loading library list...
Done


In [2]:
db.describe_table(library='crsp', table='msf')

Approximately 5153763 rows in crsp.msf.


,name,nullable,type,comment
0,cusip,True,VARCHAR(8),CUSIP Header
1,permno,True,INTEGER,PERMNO
2,permco,True,INTEGER,PERMCO
3,issuno,True,INTEGER,Nasdaq Issue Number
4,hexcd,True,SMALLINT,Exchange Code Header
5,hsiccd,True,INTEGER,Standard Industrial Classification Code Header
6,date,True,DATE,Date of Observation
7,bidlo,True,"NUMERIC(11, 5)",Bid or Low Price
8,askhi,True,"NUMERIC(11, 5)",Ask or High Price
9,prc,True,"NUMERIC(11, 5)",Price or Bid/Ask Average


In [3]:
query = """
SELECT permno, date, ret, prc, shrout
FROM crsp.msf
WHERE date >= '2020-01-01' AND date <= '2024-12-31'
"""
df = db.raw_sql(query, date_cols=['date'])
df = df.sort_values(['permno','date']).reset_index(drop=True)

print(df.shape)
print(df.head())

(546882, 5)
   permno       date       ret        prc   shrout
0   10026 2020-01-31 -0.100016     165.84  18919.0
1   10026 2020-02-28  -0.03027  160.82001  18919.0
2   10026 2020-03-31 -0.244031      121.0  18888.0
3   10026 2020-04-30  0.049835     127.03  18888.0
4   10026 2020-05-29  0.012596     128.63  18888.0


In [4]:
import numpy as np
df['logret'] = np.log(1+df['ret'])
df['mom_12_2'] = df.groupby('permno')['logret'].transform(lambda x: x.shift(1).rolling(window=11).sum())
df['mktcap'] = df['prc'].abs() * df['shrout']
print(df[['permno','date','ret','mom_12_2', 'mktcap']].head(20))


    permno       date       ret  mom_12_2         mktcap
0    10026 2020-01-31 -0.100016       NaN     3137526.96
1    10026 2020-02-28  -0.03027       NaN  3042553.76919
2    10026 2020-03-31 -0.244031       NaN      2285448.0
3    10026 2020-04-30  0.049835       NaN     2399342.64
4    10026 2020-05-29  0.012596       NaN     2429563.44
5    10026 2020-06-30 -0.007191       NaN     2401231.44
6    10026 2020-07-31 -0.031464       NaN     2326541.35
7    10026 2020-08-31  0.104118       NaN     2568775.25
8    10026 2020-09-30 -0.036668       NaN     2466326.85
9    10026 2020-10-30  0.039727       NaN  2564306.73915
10   10026 2020-11-30  0.072435       NaN     2755722.06
11   10026 2020-12-31  0.072598 -0.223327     2945193.72
12   10026 2021-01-29 -0.017442 -0.047865      2897486.8
13   10026 2021-02-26  0.039958 -0.034724   3013264.6102
14   10026 2021-03-31 -0.007275  0.284211     2988909.02
15   10026 2021-04-30  0.048271  0.228277     3133515.96
16   10026 2021-05-28  0.066642

In [5]:
db.describe_table(library='comp',table='funda')

Approximately 941532 rows in comp.funda.


,name,nullable,type,comment
0,gvkey,True,VARCHAR(7),Global Company Key
1,datadate,True,DATE,Data Date
2,fyear,True,INTEGER,Data Year - Fiscal
3,indfmt,True,VARCHAR(13),Industry Format
4,consol,True,VARCHAR(3),Level of Consolidation - Company Annual Descri...
...,...,...,...,...
944,au,True,VARCHAR(9),Auditor
945,auop,True,VARCHAR(9),Auditor Opinion
946,auopic,True,VARCHAR(2),Auditor Opinion - Internal Control
947,ceoso,True,VARCHAR(2),Chief Executive Officer SOX Certification


In [6]:
query = """
    SELECT gvkey, datadate, fyear, ceq, pstk, txditc, seq, at, lt
    FROM comp.funda
    WHERE datadate >= '2020-01-01' AND datadate <= '2024-12-31'
    AND indfmt = 'INDL'
    AND datafmt = 'STD'
    AND popsrc = 'D'
    AND consol = 'C'
"""

comp = db.raw_sql(query, date_cols=['datadate'])

print(comp.head(15))
print("shape:", comp.shape)

     gvkey   datadate  fyear      ceq  pstk  txditc      seq       at       lt
0   001004 2020-05-31   2019    902.6   0.0     0.0    902.6   2079.0   1176.4
1   001004 2021-05-31   2020    974.4   0.0     9.5    974.4   1539.7    565.3
2   001004 2022-05-31   2021   1034.5   0.0    20.0   1034.5   1573.9    539.4
3   001004 2023-05-31   2022   1099.1   0.0    33.6   1099.1   1833.1    734.0
4   001004 2024-05-31   2023   1189.8   0.0    23.9   1189.8   2770.0   1580.2
5   001019 2020-12-31   2020   13.479   0.0   0.361   13.479    40.57   27.091
6   001045 2020-12-31   2020  -6867.0   0.0     9.0  -6867.0  62008.0  68875.0
7   001045 2021-12-31   2021  -7340.0   0.0     9.0  -7340.0  66467.0  73807.0
8   001045 2022-12-31   2022  -5799.0   0.0    10.0  -5799.0  64716.0  70515.0
9   001045 2023-12-31   2023  -5202.0   0.0     9.0  -5202.0  63058.0  68260.0
10  001045 2024-12-31   2024  -3977.0   0.0     9.0  -3977.0  61783.0  65760.0
11  001050 2020-12-31   2020  202.658   0.0    6.97 

In [7]:
link_query = """
    SELECT gvkey, lpermno AS permno, linktype, linkprim, linkdt, linkenddt
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU', 'LC')
    AND linkprim IN ('P', 'C')
"""
link = db.raw_sql(link_query, date_cols=['linkdt', 'linkenddt'])

print(link.head(15))
print("shape:", link.shape)

     gvkey   permno linktype linkprim     linkdt  linkenddt
0   001000  25881.0       LU        P 1970-11-13 1978-06-30
1   001001  10015.0       LU        P 1983-09-20 1986-07-31
2   001002  10023.0       LC        C 1972-12-14 1973-06-05
3   001003  10031.0       LU        C 1983-12-07 1989-08-16
4   001004  54594.0       LU        P 1972-04-24        NaT
5   001005  61903.0       LU        C 1973-01-31 1983-01-31
6   001007  10058.0       LU        C 1973-10-01 1979-01-30
7   001007  10058.0       LU        P 1979-01-31 1984-09-28
8   001008  10066.0       LC        P 1983-08-25 1987-02-26
9   001009  10074.0       LC        C 1982-01-18 1996-03-13
10  001010  10006.0       LU        C 1950-05-01 1962-01-30
11  001010  10006.0       LU        P 1962-01-31 1984-06-28
12  001011  10082.0       LC        P 1983-03-21 1995-09-28
13  001012  10103.0       LU        P 1978-01-31 1989-12-29
14  001013  50906.0       LU        P 1979-03-16 2010-12-31
shape: (33324, 6)


In [8]:
comp_linked = comp.merge(link, on = 'gvkey', how = 'inner')

print(comp_linked.shape)
print(comp_linked[['gvkey','datadate','permno','linkdt','linkenddt']].head(15))

(38216, 14)
     gvkey   datadate   permno     linkdt  linkenddt
0   001004 2020-05-31  54594.0 1972-04-24        NaT
1   001004 2021-05-31  54594.0 1972-04-24        NaT
2   001004 2022-05-31  54594.0 1972-04-24        NaT
3   001004 2023-05-31  54594.0 1972-04-24        NaT
4   001004 2024-05-31  54594.0 1972-04-24        NaT
5   001019 2020-12-31  10189.0 1972-12-14 1988-02-09
6   001045 2020-12-31  21020.0 1950-01-01 1962-01-30
7   001045 2020-12-31  21020.0 1962-01-31 2012-01-04
8   001045 2020-12-31  21020.0 2013-12-09        NaT
9   001045 2021-12-31  21020.0 1950-01-01 1962-01-30
10  001045 2021-12-31  21020.0 1962-01-31 2012-01-04
11  001045 2021-12-31  21020.0 2013-12-09        NaT
12  001045 2022-12-31  21020.0 1950-01-01 1962-01-30
13  001045 2022-12-31  21020.0 1962-01-31 2012-01-04
14  001045 2022-12-31  21020.0 2013-12-09        NaT


In [9]:
import pandas as pd
cond_start = comp_linked['datadate'] >= comp_linked['linkdt']
cond_end = (comp_linked['datadate'] <= comp_linked['linkenddt'])|comp_linked['linkenddt'].isna()

comp_linked = comp_linked[cond_start & cond_end].copy()

print("shape:", comp_linked.shape)
print(comp_linked[['gvkey', 'datadate', 'permno', 'linkdt', 'linkenddt']].head(15))

shape: (28676, 14)
     gvkey   datadate   permno     linkdt linkenddt
0   001004 2020-05-31  54594.0 1972-04-24       NaT
1   001004 2021-05-31  54594.0 1972-04-24       NaT
2   001004 2022-05-31  54594.0 1972-04-24       NaT
3   001004 2023-05-31  54594.0 1972-04-24       NaT
4   001004 2024-05-31  54594.0 1972-04-24       NaT
8   001045 2020-12-31  21020.0 2013-12-09       NaT
11  001045 2021-12-31  21020.0 2013-12-09       NaT
14  001045 2022-12-31  21020.0 2013-12-09       NaT
17  001045 2023-12-31  21020.0 2013-12-09       NaT
20  001045 2024-12-31  21020.0 2013-12-09       NaT
21  001050 2020-12-31  11499.0 1980-11-28       NaT
22  001050 2021-12-31  11499.0 1980-11-28       NaT
23  001050 2022-12-31  11499.0 1980-11-28       NaT
24  001050 2023-12-31  11499.0 1980-11-28       NaT
25  001050 2024-12-31  11499.0 1980-11-28       NaT


In [10]:
dup_count = comp_linked.duplicated(subset=['gvkey','datadate']).sum()
print("duplicate:", dup_count)

duplicate: 0


In [11]:
comp_linked['be']=comp_linked['ceq'].fillna(0)+comp_linked['txditc'].fillna(0)-comp_linked['pstk'].fillna(0)

print("be<=0: ", (comp_linked['be']<=0).sum())
print("total number of row: ",len(comp_linked))
print(comp_linked[['gvkey','datadate','permno','ceq','txditc','pstk','be']].head(10))

be<=0:  4197
total number of row:  28676
     gvkey   datadate   permno     ceq  txditc  pstk      be
0   001004 2020-05-31  54594.0   902.6     0.0   0.0   902.6
1   001004 2021-05-31  54594.0   974.4     9.5   0.0   983.9
2   001004 2022-05-31  54594.0  1034.5    20.0   0.0  1054.5
3   001004 2023-05-31  54594.0  1099.1    33.6   0.0  1132.7
4   001004 2024-05-31  54594.0  1189.8    23.9   0.0  1213.7
8   001045 2020-12-31  21020.0 -6867.0     9.0   0.0 -6858.0
11  001045 2021-12-31  21020.0 -7340.0     9.0   0.0 -7331.0
14  001045 2022-12-31  21020.0 -5799.0    10.0   0.0 -5789.0
17  001045 2023-12-31  21020.0 -5202.0     9.0   0.0 -5193.0
20  001045 2024-12-31  21020.0 -3977.0     9.0   0.0 -3968.0


In [12]:
# eliminate observations with BE <= 0
comp_clean = comp_linked[comp_linked['be']>0].copy()
print("shape afterwards:",comp_clean.shape)

# mark datadate as the fiscal year
comp_clean['fyear_end'] = comp_clean['datadate'].dt.year

print(comp_clean[['gvkey', 'datadate', 'permno', 'be', 'fyear_end']].head(10))

shape afterwards: (24479, 15)
     gvkey   datadate   permno       be  fyear_end
0   001004 2020-05-31  54594.0    902.6       2020
1   001004 2021-05-31  54594.0    983.9       2021
2   001004 2022-05-31  54594.0   1054.5       2022
3   001004 2023-05-31  54594.0   1132.7       2023
4   001004 2024-05-31  54594.0   1213.7       2024
21  001050 2020-12-31  11499.0  209.628       2020
22  001050 2021-12-31  11499.0  212.944       2021
23  001050 2022-12-31  11499.0   221.89       2022
24  001050 2023-12-31  11499.0  241.481       2023
25  001050 2024-12-31  11499.0  259.011       2024


In [13]:
# be effective date, datadate postponed by 6 months
comp_clean['be_available'] = comp_clean['datadate'] + pd.DateOffset(months=6)

print(comp_clean[['gvkey','permno','datadate','be','be_available']].head(10))

     gvkey   permno   datadate       be be_available
0   001004  54594.0 2020-05-31    902.6   2020-11-30
1   001004  54594.0 2021-05-31    983.9   2021-11-30
2   001004  54594.0 2022-05-31   1054.5   2022-11-30
3   001004  54594.0 2023-05-31   1132.7   2023-11-30
4   001004  54594.0 2024-05-31   1213.7   2024-11-30
21  001050  11499.0 2020-12-31  209.628   2021-06-30
22  001050  11499.0 2021-12-31  212.944   2022-06-30
23  001050  11499.0 2022-12-31   221.89   2023-06-30
24  001050  11499.0 2023-12-31  241.481   2024-06-30
25  001050  11499.0 2024-12-31  259.011   2025-06-30


In [14]:
# sort values on date
df_sorted = df.sort_values('date').copy()
comp_sorted = comp_clean.sort_values('be_available').copy()

df_sorted['permno'] = df_sorted['permno'].astype('Int64')
comp_sorted['permno'] = comp_sorted['permno'].astype('Int64')

# match crsp with be, be_available <= date
merged = pd.merge_asof(
    df_sorted,
    comp_sorted[['permno', 'be_available', 'be']],
    left_on = 'date',
    right_on = 'be_available',
    by = 'permno',
    direction = 'backward'
)

print(merged[['permno', 'date', 'ret', 'mktcap', 'be', 'be_available']].head(15))
print(merged[['permno', 'date', 'ret', 'mktcap', 'be', 'be_available']].tail(15))
print("shape:", merged.shape)

    permno       date       ret      mktcap    be be_available
0    10026 2020-01-31 -0.100016  3137526.96  <NA>          NaT
1    17623 2020-01-31 -0.035591    97947.75  <NA>          NaT
2    17622 2020-01-31   0.00589   32543.925  <NA>          NaT
3    12068 2020-01-31 -0.020776  3027757.14  <NA>          NaT
4    17621 2020-01-31  0.008275    108805.0  <NA>          NaT
5    17620 2020-01-31 -0.030425     43198.7  <NA>          NaT
6    91612 2020-01-31 -0.082321  690986.828  <NA>          NaT
7    17615 2020-01-31  -0.02667     5793.75  <NA>          NaT
8    17614 2020-01-31   0.02042     71775.0  <NA>          NaT
9    17613 2020-01-31 -0.017493   12661.695  <NA>          NaT
10   91614 2020-01-31 -0.000625   629324.36  <NA>          NaT
11   17610 2020-01-31       0.0    125792.0  <NA>          NaT
12   91615 2020-01-31  -0.07143      6370.0  <NA>          NaT
13   17608 2020-01-31  0.016633     83274.4  <NA>          NaT
14   17607 2020-01-31  0.006671    131970.0  <NA>      

In [15]:
print("total rows of na:", merged['be'].notna().sum())
print("total rows:", len(merged))

one = merged[merged['permno'] == 10026].sort_values('date')
print(one[['date', 'mktcap', 'be', 'be_available']].iloc[[0, 6, 12, 18, 24, 30]])

total rows of na: 212027
total rows: 546882
             date         mktcap       be be_available
0      2020-01-31     3137526.96     <NA>          NaT
50628  2020-07-31     2326541.35     <NA>          NaT
96859  2021-01-29      2897486.8     <NA>          NaT
147943 2021-07-30     3133740.32  873.911   2021-03-30
207947 2022-01-31      2898795.9  873.911   2021-03-30
259298 2022-07-29  2600707.72808  907.232   2022-03-30


In [16]:
# B/M
merged['bm'] = merged['be'] / merged['mktcap']

print(merged[['permno', 'date', 'mktcap', 'be', 'bm']].dropna(subset = ['bm']).head(10))
print("row number of bm:", merged['bm'].notna().sum())

       permno       date       mktcap        be        bm
46263   34948 2020-07-31    717870.92   545.138  0.000759
46358   17382 2020-07-31    1162836.0   166.228  0.000143
46451   83011 2020-07-31   6774364.08  1238.615  0.000183
46618   88360 2020-07-31  24435519.99  8709.813  0.000356
46765   80432 2020-07-31    1655000.0  1247.853  0.000754
47026   14141 2020-07-31   4648357.62    1417.0  0.000305
47075   14609 2020-07-31    701379.19    110.83  0.000158
47220   14296 2020-07-31     53884.87    130.78  0.002427
47398   14544 2020-07-31    281046.48   406.033  0.001445
47437   14803 2020-07-31   3963520.32  1160.787  0.000293
row number of bm: 210789


In [17]:
# construct returns next month
# sort on stocks and time
merged = merged.sort_values(['permno', 'date']).reset_index(drop=True)

# target: return next month
merged['ret_next'] =  merged.groupby('permno')['ret'].shift(-1)

# check
check = merged[merged['permno']==10026][['date', 'ret', 'ret_next']].head(8)
print(check)

        date       ret  ret_next
0 2020-01-31 -0.100016  -0.03027
1 2020-02-28  -0.03027 -0.244031
2 2020-03-31 -0.244031  0.049835
3 2020-04-30  0.049835  0.012596
4 2020-05-29  0.012596 -0.007191
5 2020-06-30 -0.007191 -0.031464
6 2020-07-31 -0.031464  0.104118
7 2020-08-31  0.104118 -0.036668


In [18]:
# the final panel for forecasting
panel = merged[['permno', 'date', 'mktcap', 'mom_12_2', 'bm', 'ret_next']].copy()

panel = panel.dropna(subset=['mktcap', 'mom_12_2', 'bm', 'ret_next'])

print(panel.head(10))
print("shape:", panel.shape)
print("time span:", panel['date'].min(), panel['date'].max())

    permno       date         mktcap  mom_12_2        bm  ret_next
14   10026 2021-03-31     2988909.02  0.284211  0.000292  0.048271
15   10026 2021-04-30     3133515.96  0.228277  0.000279  0.066642
16   10026 2021-05-28     3342340.88  0.262902  0.000261 -0.003058
17   10026 2021-06-30     3324429.01  0.334634  0.000263 -0.057508
18   10026 2021-07-30     3133740.32  0.363541  0.000279 -0.003772
19   10026 2021-08-31  3121920.44936  0.205266   0.00028  -0.06294
20   10026 2021-09-30  2916417.07084  0.238844    0.0003 -0.034485
21   10026 2021-10-29      2815844.2  0.134878   0.00031 -0.074348
22   10026 2021-11-30      2606629.3  0.029853  0.000335  0.161173
23   10026 2021-12-31  3015298.63089 -0.117488   0.00029 -0.039694
shape: (201001, 6)
time span: 2020-12-31 00:00:00 2024-11-29 00:00:00


In [19]:
print(panel[['mktcap', 'mom_12_2', 'bm', 'ret_next']].describe())

print("number of stocks per month: ")
print(panel.groupby('date').size().head())

                mktcap       mom_12_2        bm  ret_next
count         201001.0  201001.000000  201001.0  201001.0
mean   10660148.858425      -0.160306  0.002567  0.000247
std    73280580.043777       0.726207  0.025794  0.256273
min            90.3264      -8.959638       0.0 -0.968292
25%           129560.4      -0.403496  0.000262 -0.089936
50%          798857.06      -0.026900  0.000584  -0.00939
75%         4041676.29       0.241698  0.001135  0.069098
max      3587438272.59       4.502891  1.698489      39.0
number of stocks per month: 
date
2020-12-31    515
2021-01-29    513
2021-02-26    540
2021-03-31    746
2021-04-30    784
dtype: int64


In [20]:
# figure out mktcap units
big = merged[merged['permno']==10107][['date', 'mktcap']].tail(3)
print("Microsoft mktcap:")
print(big)

Microsoft mktcap:
          date            mktcap
511 2024-10-31  3021163968.69881
512 2024-11-29  3148374633.91119
513 2024-12-31      3133802341.5


In [21]:
# clean the stock universe
# price filter and market-cap filter
clean = merged[(merged['prc'].abs() >= 5) & (merged['mktcap'] >= 1_000_000)].copy()

print("before universal filter: ", merged.shape)
print("after universal filter: ", clean.shape)

#rebuild the panel
panel = clean[['permno', 'date', 'mktcap', 'mom_12_2', 'bm', 'ret_next']].copy()
panel = panel.dropna(subset=['mktcap', 'mom_12_2', 'bm', 'ret_next'])

print("Clean panel shape:", panel.shape)
print(panel[['mktcap', 'mom_12_2', 'bm', 'ret_next']].describe())

before universal filter:  (546882, 12)
after universal filter:  (175136, 12)
Clean panel shape: (91773, 6)
                 mktcap      mom_12_2        bm  ret_next
count           91773.0  91773.000000   91773.0   91773.0
mean    23004228.150158      0.093045  0.000811  0.003758
std    107148995.916005      0.408152   0.00338  0.132025
min           1000163.7     -3.531928       0.0 -0.886269
25%           2164388.1     -0.117134  0.000177 -0.064275
50%          4702834.92      0.093001  0.000371 -0.000474
75%          14169424.9      0.307755   0.00069  0.065371
max       3587438272.59      4.502891  0.099718  16.25053


In [22]:
# winsorize extreme returns
print("ret_next > 1:", (panel['ret_next'] > 1).sum())
print("ret_next > 0.5", (panel['ret_next'] > 0.5).sum())
print(panel['ret_next'].quantile([0.001, 0.01, 0.99, 0.999]))

ret_next > 1: 31
ret_next > 0.5 264
0.001    -0.48344
0.010   -0.288681
0.990    0.337942
0.999    0.696207
Name: ret_next, dtype: Float64


In [23]:
# winsorize
lower = panel['ret_next'].quantile(0.001)
upper = panel['ret_next'].quantile(0.999)
panel['ret_next']=panel['ret_next'].clip(lower, upper)

print(f"Clipping ret_next to [{lower:.4f}, {upper:.4f}]")
print(panel['ret_next'].describe())

Clipping ret_next to [-0.4834, 0.6962]
count     91773.0
mean     0.003408
std      0.117516
min      -0.48344
25%     -0.064275
50%     -0.000474
75%      0.065371
max      0.696207
Name: ret_next, dtype: Float64


In [24]:
#cross-sectional rank standardization:
feature_cols = ['mktcap', 'mom_12_2', 'bm']

def rank_normalize(s):
    return s.rank(pct=True) - 0.5

for col in feature_cols:
    panel[col + '_rank'] = panel.groupby('date')[col].transform(rank_normalize)

# check the result
print(panel[['date', 'mktcap', 'mktcap_rank', 'mom_12_2', 'mom_12_2_rank', 'bm', 'bm_rank']].head())
print(panel[[c + '_rank' for c in feature_cols]].describe())


         date      mktcap  mktcap_rank  mom_12_2  mom_12_2_rank        bm  \
14 2021-03-31  2988909.02    -0.173522  0.284211      -0.302057  0.000292   
15 2021-04-30  3133515.96    -0.176399  0.228277      -0.315085  0.000279   
16 2021-05-28  3342340.88    -0.146635  0.262902      -0.204327  0.000261   
17 2021-06-30  3324429.01    -0.119731  0.334634      -0.112108  0.000263   
18 2021-07-30  3133740.32    -0.141270  0.363541      -0.025170  0.000279   

     bm_rank  
14  0.057841  
15  0.052311  
16  0.024038  
17 -0.059193  
18 -0.041497  
        mktcap_rank  mom_12_2_rank       bm_rank
count  91773.000000   91773.000000  91773.000000
mean       0.000262       0.000262      0.000262
std        0.288677       0.288677      0.288677
min       -0.499560      -0.499560     -0.499560
25%       -0.249768      -0.249768     -0.249768
50%        0.000233       0.000233      0.000233
75%        0.250242       0.250242      0.250242
max        0.500000       0.500000      0.500000


In [25]:
from sklearn.linear_model import LinearRegression
import numpy as np

#features, x and y
feature_ranks = ['mktcap_rank', 'mom_12_2_rank', 'bm_rank']
X = panel[feature_ranks]
y = panel['ret_next']

# fit an OLS
model = LinearRegression()
model.fit(X, y)

# print output
for feat, coef in zip(feature_ranks, model.coef_):
    print(f"{feat} coef = {coef:+.5f}")

print(f"intercept = {model.intercept_:+.5f}")
print(f"R-squared: {model.score(X, y):.5f}")




mktcap_rank coef = +0.00717
mom_12_2_rank coef = +0.01170
bm_rank coef = +0.01146
intercept = +0.00340
R-squared: 0.00152


In [26]:
from sklearn.linear_model import LinearRegression

feature_ranks = ['mktcap_rank', 'mom_12_2_rank', 'bm_rank']

#split: train on data up to end-2023
split_date = '2024-01-01'
train = panel[panel['date'] < split_date]
test = panel[panel['date'] >= split_date]

print("Train period:", train['date'].min(), "->", train['date'].max(), "rows:", len(train))
print("Test period:", test['date'].min(), "->", test['date'].max(), "rows:", len(test))

# fit on training data
model = LinearRegression()
model.fit(train[feature_ranks], train['ret_next'])

# predict on the test set
test = test.copy()
test['pred'] = model.predict(test[feature_ranks])

print("Coefficients:")
for feat, coef in zip(feature_ranks, model.coef_):
    print(f"{feat:20s}{coef:+.5f}")


Train period: 2020-12-31 00:00:00 -> 2023-12-29 00:00:00 rows: 68450
Test period: 2024-01-31 00:00:00 -> 2024-11-29 00:00:00 rows: 23323
Coefficients:
mktcap_rank         +0.00880
mom_12_2_rank       +0.01027
bm_rank             +0.01647


In [27]:
from scipy.stats import spearmanr
import numpy as np

# IC: spearman
def monthly_ic(group):
    ic, _ = spearmanr(group['pred'], group['ret_next'])
    return ic

ic_by_month = test.groupby('date').apply(monthly_ic, include_groups=False)

print("Monthly ICs:")
print(ic_by_month)

mean_ic = ic_by_month.mean()
std_ic = ic_by_month.std()
n = ic_by_month.count()
t_stat = mean_ic / (std_ic/ np.sqrt(n))

print("mean IC:", mean_ic)
print("std IC:", std_ic)
print("IC t-stat:", t_stat)
print("months:", n)

Monthly ICs:
date
2024-01-31   -0.045943
2024-02-29    0.174436
2024-03-28    0.119687
2024-04-30    0.080387
2024-05-31   -0.112939
2024-06-28    0.032309
2024-07-31    0.105802
2024-08-30    0.005038
2024-09-30    0.037050
2024-10-31   -0.065253
2024-11-29   -0.007689
dtype: float64
mean IC: 0.029353248816799262
std IC: 0.08634607658004056
IC t-stat: 1.127482759601965
months: 11


In [28]:
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

feature_ranks = ['mktcap_rank', 'mom_12_2_rank', 'bm_rank']

# all unique months, sorted
months = np.sort(panel['date'].unique())
min_train_months = 18
records = []

for i in range(min_train_months, len(months)):
    target_month = months[i]

    # train and test
    train = panel[panel['date'] < target_month]
    test_m = panel[panel['date'] == target_month]

    # fit on past
    model = LinearRegression()
    model.fit(train[feature_ranks], train['ret_next'])
    preds = model.predict(test_m[feature_ranks])

    # this month's IC
    ic, _ = spearmanr(preds, test_m['ret_next'])
    records.append((target_month, ic))

# collect into a series
ic_df = pd.DataFrame(records, columns=['date', 'ic']).set_index('date')
print("Number of out-of-sample months:", len(ic_df))
print(ic_df.head())
print(ic_df.tail())




Number of out-of-sample months: 30
                  ic
date                
2022-06-30 -0.164122
2022-07-29  0.049533
2022-08-31  0.005082
2022-09-30  0.170968
2022-10-31  0.086917
                  ic
date                
2024-07-31  0.111897
2024-08-30  0.012720
2024-09-30  0.070983
2024-10-31 -0.038015
2024-11-29 -0.057331


In [29]:
# verify no look-ahead
target = months[20]
train_check = panel[panel['date'] < target]

print("target month being predicted:", target)
print("Latest date in training date:", train_check['date'].max())
print("Is training strictly before target?", train_check['date'].max() < target)

target month being predicted: 2022-08-31T00:00:00.000000000
Latest date in training date: 2022-07-29 00:00:00
Is training strictly before target? True


In [30]:
mean_ic = ic_df['ic'].mean()
std_ic = ic_df['ic'].std()
n = len(ic_df)
t_stat = mean_ic/(std_ic/np.sqrt(n))
hit_rate = (ic_df['ic'] > 0).mean()

print(f"mean IC: {mean_ic:+.4f}")
print(f"std IC: {std_ic:.4f}")
print(f"IC t-stat:  {t_stat:+.2f}")
print(f"Hit rate: {hit_rate:.1%}")
print(f"Months: {n}")

mean IC: +0.0065
std IC: 0.1062
IC t-stat:  +0.34
Hit rate: 63.3%
Months: 30


In [32]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

feature_banks = ['mktcap_rank', 'mom_12_2_rank', 'bm_rank']
months= np.sort(panel['date'].unique())
min_train_months = 18

def run_walkforward(model_fn):
    """model_fn returns a gresh model each month. return series of montly ICs."""
    records = []
    for i in range(min_train_months, len(months)):
        target_month = months[i]
        train = panel[panel['date'] < target_month]
        test_m = panel[panel['date'] == target_month]

        model = model_fn()
        model.fit(train[feature_banks], train['ret_next'])
        preds = model.predict(test_m[feature_ranks])

        ic, _ = spearmanr(preds, test_m['ret_next'])
        records.append((target_month, ic))
    return pd.DataFrame(records, columns=['date', 'ic']).set_index('date')['ic']

def summarize(ic_series, label):
    m, s, n = ic_series.mean(), ic_series.std(), len(ic_series)
    t = m / (s / np.sqrt(n))
    hit = (ic_series > 0).mean()
    print(f"{label:22s} mean IC {m:+.4f}, t {t:+.2f}, hit {hit:.1%}, n{n}")

# Ridge with a fixed alpha, run through the same walk-forward
ic_ridge = run_walkforward(lambda: Ridge(alpha = 1.0))
ic_ols = run_walkforward(lambda: LinearRegression())
summarize(ic_ols, "OLS")
summarize(ic_ridge, "Ridge (alpha = 1.o)")


OLS                    mean IC +0.0065, t +0.34, hit 63.3%, n30
Ridge (alpha = 1.o)    mean IC +0.0065, t +0.34, hit 63.3%, n30


In [34]:
# tune alpha
def run_walkforward_ridge_turned(alphas):
    records = []
    for i in range(min_train_months, len(months)):
        target_month = months[i]
        train_all = panel[panel['date'] < target_month]

        # inner tuning:
        train_months = np.sort(train_all['date'].unique())
        val_cutoff = train_months[-6]
        fit_part = train_all[train_all['date'] < val_cutoff]
        val_part = train_all[train_all['date'] >= val_cutoff]

        # try each alpha
        best_alpha, best_ic = None, -np.inf
        for a in alphas:
            m = Ridge(alpha=a)
            m.fit(fit_part[feature_ranks], fit_part['ret_next'])
            val_pred = m.predict(val_part[feature_ranks])
            val_ic, _ = spearmanr(val_pred, val_part['ret_next'])
            if val_ic > best_ic:
                best_ic, best_alpha = val_ic, a

        # refit all past with the chosen alpha
        final = Ridge(alpha = best_alpha)
        final.fit(train_all[feature_ranks], train_all['ret_next'])
        test_m = panel[panel['date'] == target_month]
        preds = final.predict(test_m[feature_ranks])

        ic, _ = spearmanr(preds, test_m['ret_next'])
        records.append((target_month, ic, best_alpha))
    return pd.DataFrame(records, columns=['date', 'ic', 'alpha']).set_index('date')

alphas = [0.01, 0.1, 1, 10, 100]
res_tuned = run_walkforward_ridge_turned(alphas)
summarize(res_tuned['ic'], "Ridge (tuned)")
print("Chosen alpha distribution:")
print(res_tuned['alpha'].value_counts())

    

Ridge (tuned)          mean IC +0.0065, t +0.33, hit 63.3%, n30
Chosen alpha distribution:
alpha
0.01      11
100.00     8
0.10       5
1.00       4
10.00      2
Name: count, dtype: int64


In [35]:
# sanity check on whether  validation was strictly before target month
i = 25
target_month = months[i]
train_all = panel[panel['date'] < target_month]
train_months = np.sort(train_all['date'].unique())
val_cutoff = train_months[-6]
print("target month: ", target_month)
print("validation cutoff: ", val_cutoff)
print("val slice before target?", val_cutoff < target_month)
print("latest val month < target?", train_months[-1] < target_month)

target month:  2023-01-31T00:00:00.000000000
validation cutoff:  2022-07-29T00:00:00.000000000
val slice before target? True
latest val month < target? True
